### Dupla

Thiago Ramon Martins Barros - 2415310038
Ricardo da Silva Santiago - 2415310023



## Parte 1

In [1]:

import numpy as np
import matplotlib.pyplot as plt
from prettytable import PrettyTable
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [2]:
dados = np.load("dataAll20262.txt")

print("Dimensão original:", dados.shape)

dados = dados.reshape(-1, 3)

X = dados[:, :2]
y = dados[:, 2].astype(int)

print("Entradas:", X.shape)
print("Rótulos:", y.shape)
print("Classes:", np.unique(y, return_counts=True))

FileNotFoundError: [Errno 2] No such file or directory: 'dataAll20262.txt'

In [ ]:
def degrau(u):
    return 1 if u >= 0 else 0

In [ ]:

def treinar_perceptron(
    X, y,
    eta=0.1,
    intervalo=(-0.5, 0.5),
    seed=42,
    verbose=True,
    max_epocas=None,
    embaralhar=False,
    registrar_erros=False
):
    rng = np.random.default_rng(seed)

    X_bias = np.column_stack((np.ones(len(X)), X))

    pesos = rng.uniform(
        intervalo[0], intervalo[1],
        size=X_bias.shape[1]
    )

    if verbose:
        print("Pesos iniciais:", pesos.copy())

    epocas = 0
    ajustes = 0
    historico_erros = []

    while max_epocas is None or epocas < max_epocas:
        erros = 0
        epocas += 1

        if embaralhar:
            indices = rng.permutation(len(X_bias))
        else:
            indices = np.arange(len(X_bias))

        for i in indices:
            xi = X_bias[i]
            yd = y[i]

            u = np.dot(pesos, xi)
            previsao = degrau(u)

            erro = yd - previsao

            if erro != 0:
                pesos += eta * erro * xi
                ajustes += 1
                erros += 1

        if registrar_erros:
            historico_erros.append(erros)

        if erros == 0 and max_epocas is None:
            break

    if registrar_erros:
        return pesos, epocas, ajustes, np.array(historico_erros)

    return pesos, epocas, ajustes


In [ ]:
pesos, epocas, ajustes = treinar_perceptron(X, y)

print("Pesos finais:", pesos)
print("Número de épocas:", epocas)
print("Total de ajustes:", ajustes)

In [ ]:

plt.figure(figsize=(9, 6))

plt.scatter(
    X[y == 0, 0], X[y == 0, 1],
    color="red", s=12, alpha=0.7, label="Classe 0"
)

plt.scatter(
    X[y == 1, 0], X[y == 1, 1],
    color="blue", s=12, alpha=0.7, label="Classe 1"
)

x_reta = np.linspace(X[:, 0].min(), X[:, 0].max(), 200)
y_reta = -(pesos[0] + pesos[1] * x_reta) / pesos[2]

plt.plot(x_reta, y_reta, color="green",
         linewidth=2, label="Reta de decisão")

plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Classificação com Perceptron")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## Parte 2

In [ ]:
dados2 = np.load("data320262.txt")

print("Dimensão original:", dados2.shape)

In [ ]:
dados2 = dados2.reshape(-1, 3)

X2 = dados2[:, :2]
y2 = dados2[:, 2].astype(int)

print("Entradas:", X2.shape)
print("Rótulos:", y2.shape)
print("Classes:", np.unique(y2, return_counts=True))

In [ ]:
n_repeticoes = 10
resultados = {}

for eta in [0.4, 0.1, 0.01]:
    for intervalo in [(-100, 100), (-0.5, 0.5)]:
        ajustes_exec = np.zeros(n_repeticoes, dtype=int)
        epocas_exec = np.zeros(n_repeticoes, dtype=int)

        for i in range(n_repeticoes):
            _, epocas_exec[i], ajustes_exec[i] = treinar_perceptron(
                X2, y2, eta=eta, intervalo=intervalo, seed=i, verbose=False
            )

        resultados[(eta, intervalo)] = (ajustes_exec, epocas_exec)

        print(f"Aprendizado: {eta}, Intervalo: {intervalo}")
        print(f"Total de ajustes: {ajustes_exec}")
        print(f"Total de épocas: {epocas_exec}")

In [ ]:
from prettytable import PrettyTable

tabela = PrettyTable()
tabela.field_names = ["Taxa de Aprendizado", "Intervalo de Pesos", "Qtde Ajustes", "Min Épocas", "Max Épocas"]

for (eta, intervalo), (ajustes_exec, epocas_exec) in resultados.items():
    tabela.add_row([
        f"η = {eta}",
        f"({intervalo[0]}, +{intervalo[1]})",
        f"{ajustes_exec.mean():.2f} ± {ajustes_exec.std(ddof=1):.2f}",
        epocas_exec.min(),
        epocas_exec.max(),
    ])

print(tabela)

In [ ]:
rotulos = [f"η={eta}\nI={intervalo}" for eta, intervalo in resultados]
ajustes_todos = [ajustes_exec for ajustes_exec, _ in resultados.values()]
epocas_todos = [epocas_exec for _, epocas_exec in resultados.values()]
posicoes = np.arange(1, len(rotulos) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.boxplot(ajustes_todos, showmeans=True)
ax1.set_xticks(posicoes, rotulos)
ax1.set_title("Quantidade de ajustes no vetor de pesos")
ax1.set_ylabel("Ajustes")
ax1.grid(alpha=0.2)

ax2.boxplot(epocas_todos, showmeans=True)
ax2.set_xticks(posicoes, rotulos)
ax2.set_title("Épocas até a convergência")
ax2.set_ylabel("Épocas")
ax2.grid(alpha=0.2)

fig.suptitle("Perceptron - 10 repetições por configuração (triângulo = média)")
plt.tight_layout()
plt.show()

### Análise dos resultados

Todas as 60 execuções convergiram, então o problema é linearmente separável. Mesmo assim, cada execução precisou de milhares de ajustes e de milhares de épocas. Portanto, não é possível identificar uma melhor ou pior, elas são equivalentes.

- A variação de cada configuração é muito grande, com cada desvio padrão dos ajustes chegando a cerca de 1/3 da média. Entre a execução mais rápida e a mais lenta de uma mesma configuração, o número de épocas varia de 2 a 5 vezes.
- Os bloxplots se sobrepôem, isto pois, a diferença entre as médias é da mesma ordem do desvio padrão. Com apenas 10 repetições, essas diferenças podem ser explicadas pelo sorteio dos pesos iniciais. Para afirmar com rigor que uma configuração é melhor, seria necessário um teste estatístico.

**Por que as configurações se comportam de forma parecida?**

1. **η só muda a escala dos pesos.** A função degrau depende apenas do sinal de $u$. Dividindo os pesos por η, a regra de ajuste vira $\mathbf{w}' \leftarrow \mathbf{w}' + e\,\mathbf{x}$, sem η. Ou seja, η não muda as previsões nem a reta encontrada. O que importa é apenas a razão $\mathbf{w}(0)/\eta$.
2. **A escala dos dados domina o treinamento.** $x_1$ e $x_2$ vão de −1000 a +1000, mas a entrada do bias vale 1. A cada ajuste, $w_1$ e $w_2$ mudam até ±1000·η, enquanto o bias muda só ±η. Por isso são necessários muitos ajustes até o bias atingir o valor que a reta exige. No final, os pesos são muito maiores que os iniciais, e o intervalo de inicialização, seja (−0.5, +0.5) ou (−100, +100), pesa pouco. Normalizar as entradas reduziria bastante o número de épocas.

## Parte 3

In [ ]:

dados3 = np.load("dataHoldout20262.txt")

print("Dimensão original:", dados3.shape)

dados3 = dados3.reshape(-1, 3)

X3 = dados3[:, :2]
y3 = dados3[:, 2].astype(int)

print("Entradas:", X3.shape)
print("Rótulos:", y3.shape)
print("Classes:", np.unique(y3, return_counts=True))


In [ ]:

plt.figure(figsize=(9, 6))

plt.scatter(
    X3[y3 == 0, 0], X3[y3 == 0, 1],
    color="red", s=12, alpha=0.7, label="Classe 0"
)

plt.scatter(
    X3[y3 == 1, 0], X3[y3 == 1, 1],
    color="blue", s=12, alpha=0.7, label="Classe 1"
)

plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Parte III - Distribuição inicial dos dados")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


In [ ]:

rng = np.random.default_rng(3)

indices = rng.permutation(len(X3))

n_treino = int(0.7 * len(X3))

indices_treino = indices[:n_treino]
indices_teste = indices[n_treino:]

X_treino = X3[indices_treino]
y_treino = y3[indices_treino]

X_teste = X3[indices_teste]
y_teste = y3[indices_teste]

print("Treinamento:", X_treino.shape, y_treino.shape)
print("Teste:", X_teste.shape, y_teste.shape)

print("Classes no treinamento:",
      np.unique(y_treino, return_counts=True))

print("Classes no teste:",
      np.unique(y_teste, return_counts=True))


In [ ]:

pesos3, epocas3, ajustes3, erros3 = treinar_perceptron(
    X_treino,
    y_treino,
    eta=0.1,
    intervalo=(-0.5, 0.5),
    seed=3,
    max_epocas=100,
    embaralhar=True,
    registrar_erros=True
)

print("Pesos finais:", pesos3)
print("Total de épocas:", epocas3)
print("Total de ajustes:", ajustes3)
print("Erros na primeira época:", erros3[0])
print("Erros na última época:", erros3[-1])

In [ ]:

plt.figure(figsize=(9, 5))

plt.plot(
    np.arange(1, epocas3 + 1),
    erros3,
    color="purple",
    linewidth=1.0
)

plt.xlabel("Época")
plt.ylabel("Quantidade de erros")
plt.title("Parte III - Erros durante o treinamento")
plt.grid(alpha=0.3)
plt.show()


### Avaliação do Perceptron

In [ ]:

def prever_perceptron(X, pesos):
    X_bias = np.column_stack((np.ones(len(X)), X))

    u = np.dot(X_bias, pesos)

    return (u >= 0).astype(int)


y_pred_treino = prever_perceptron(X_treino, pesos3)
y_pred_teste = prever_perceptron(X_teste, pesos3)

print("Previsões no treinamento:", len(y_pred_treino))
print("Previsões no teste:", len(y_pred_teste))
print("Classes previstas no teste:",
      np.unique(y_pred_teste, return_counts=True))

In [ ]:
matriz_confusao = confusion_matrix(
    y_teste,
    y_pred_teste,
    labels=[0, 1]
)

print("Matriz de confusão:")
print(matriz_confusao)

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz_confusao,
    display_labels=[0, 1]
)

disp.plot(cmap="Blues", values_format="d")
plt.title("Parte III - Matriz de Confusão (Teste)")
plt.show()


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

acuracia = accuracy_score(y_teste, y_pred_teste)
precisao = precision_score(y_teste, y_pred_teste, zero_division=0)
revocacao = recall_score(y_teste, y_pred_teste, zero_division=0)
f1 = f1_score(y_teste, y_pred_teste, zero_division=0)

print("Métricas no conjunto de teste:")
print(f"Acurácia:  {acuracia:.4f} ({acuracia * 100:.2f}%)")
print(f"Precisão: {precisao:.4f}")
print(f"Recall: {revocacao:.4f}")
print(f"F1-Score: {f1:.4f}")


In [ ]:

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(16, 6),
    sharex=True, sharey=True
)

x_reta = np.linspace(X3[:, 0].min(), X3[:, 0].max(), 200)

if np.isclose(pesos3[2], 0):
    x_vertical = -pesos3[0] / pesos3[1]
else:
    y_reta = -(pesos3[0] + pesos3[1] * x_reta) / pesos3[2]

conjuntos = [
    (ax1, X_treino, y_treino, "Treinamento (70%)"),
    (ax2, X_teste, y_teste, "Teste (30%)")
]

for ax, X_atual, y_atual, titulo in conjuntos:
    ax.scatter(
        X_atual[y_atual == 0, 0],
        X_atual[y_atual == 0, 1],
        color="red", s=12, alpha=0.7, label="Classe 0"
    )

    ax.scatter(
        X_atual[y_atual == 1, 0],
        X_atual[y_atual == 1, 1],
        color="blue", s=12, alpha=0.7, label="Classe 1"
    )

    if np.isclose(pesos3[2], 0):
        ax.axvline(
            x_vertical, color="green",
            linewidth=2, label="Reta de decisão"
        )
    else:
        ax.plot(
            x_reta, y_reta,
            color="green", linewidth=2,
            label="Reta de decisão"
        )

    ax.set_title(titulo)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_xlim(X3[:, 0].min(), X3[:, 0].max())
    ax.set_ylim(X3[:, 1].min(), X3[:, 1].max())
    ax.legend()
    ax.grid(alpha=0.2)

fig.suptitle("Parte III - Solução do Perceptron")
plt.tight_layout()
plt.show()


## Análise dos resultados

O treinamento do Perceptron foi realizado durante 100 épocas, utilizando 70% dos exemplos para treinamento e 30% para teste. Os exemplos de treinamento foram apresentados em ordem aleatória a cada época, conforme solicitado.

A matriz de confusão obtida no conjunto de teste apresentou 122 verdadeiros negativos, 105 verdadeiros positivos, 25 falsos positivos e 48 falsos negativos. Portanto, o modelo classificou corretamente 227 dos 300 exemplos avaliados.

A acurácia obtida foi de 75,67%, indicando que aproximadamente três quartos dos exemplos de teste foram classificados corretamente. A precisão de 80,77% demonstra que a maioria das previsões positivas estava correta, enquanto a revocação de 68,63% evidencia que o modelo deixou de identificar uma parcela relevante dos exemplos pertencentes à classe 1. O F1-Score de 74,20% representa o equilíbrio entre precisão e revocação.
Os gráficos de treinamento e teste demonstram que a reta de decisão obtida consegue separar parcialmente as classes, mas apresenta erros em determinadas regiões do espaço de atributos. Esse comportamento é compatível com a natureza não linearmente separável do problema.

Assim, considerando a simplicidade do modelo, a solução apresentou desempenho razoável. Entretanto, os resultados demonstram as limitações de um único neurônio Perceptron para problemas que não podem ser perfeitamente separados por uma fronteira linear. Concluindo assim que, embora o algoritmo tenha produzido uma solução capaz de classificar corretamente a maioria dos exemplos de teste, modelos com fronteiras de decisão não lineares poderiam ser investigados para buscar melhores resultados.